# Localized-adjuvant vs metastatic ADT survival results

Read-only comparison of the six result trees produced by `09_adt_intent_survival.ipynb`: two medication-derived ADT-intent strata crossed with platinum, NEPC, and AVPC. This notebook does not rebuild cohorts or refit models.

Three interpretation constraints apply throughout:

1. `ADT_INTENT` is a retrospective medication-history stratum, not a metastatic-status label available prospectively at the landmark.
2. Localized and metastatic runs use disjoint patients and independently derived splits/features. Differences can reflect cohort size, event count, case mix, or biology.
3. “Significant in one stratum but not the other” is not itself evidence of different effects. The univariate section therefore reports an approximate Wald heterogeneity test for the difference between the two log hazard ratios.

In [ ]:
from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, ".")
import compass_pipeline as cp

DATA_ROOT = cp._PROFILE_OUTPUT_ROOT
SURVIVAL_ROOT = DATA_ROOT / "survival_analysis"
ENDPOINTS = ("platinum", "nepc", "avpc")
LANDMARKS = (0, 90, 180)
ID_COL = "DFCI_MRN"
STRATA = {
    "localized": {"label": "adt_localized", "title": "Localized-adjuvant", "color": "#0b6ba8"},
    "metastatic": {"label": "adt_metastatic", "title": "Metastatic", "color": "#c1272d"},
}
EVENT_COLS = {"platinum": "PLATINUM", "nepc": "NEPC", "avpc": "AVPC"}
DURATION_COLS = {"platinum": "t_platinum", "nepc": "t_nepc", "avpc": "t_avpc"}

def endpoint_suffix(endpoint):
    return "" if endpoint == "platinum" else f"_{endpoint}"

def tree(stratum, endpoint, kind):
    label = STRATA[stratum]["label"]
    prefix = "prediction_inputs" if kind == "inputs" else "local_runs"
    return SURVIVAL_ROOT / f"{prefix}_{label}{endpoint_suffix(endpoint)}"

MISSING = []
def require(path, what):
    if Path(path).exists():
        return True
    MISSING.append(f"{what}: {path}")
    return False

print(f"data root: {DATA_ROOT}")

## 1. Artifact inventory

In [ ]:
inventory = []
for endpoint in ENDPOINTS:
    for stratum in STRATA:
        for kind in ("inputs", "results"):
            path = tree(stratum, endpoint, kind)
            inventory.append({
                "endpoint": endpoint,
                "stratum": stratum,
                "artifact": kind,
                "exists": path.exists(),
                "path": str(path),
            })
inventory = pd.DataFrame(inventory)
display(inventory)

## 2. Modelled cohort and event counts

These are the actual per-landmark Stage-3 cohorts after endpoint eligibility, PSA/PARPi filters, and lab availability. Sparse-event warnings should govern how much weight is placed on every downstream comparison.

In [ ]:
cohort_rows = []
cohort_ids = {}
for endpoint in ENDPOINTS:
    for stratum in STRATA:
        for landmark in LANDMARKS:
            path = tree(stratum, endpoint, "inputs") / f"aggregated_landmark{landmark}.csv"
            base = {"endpoint": endpoint, "stratum": stratum, "landmark_days": landmark}
            if not require(path, f"{stratum}/{endpoint} aggregated landmark {landmark}"):
                cohort_rows.append({**base, "status": "missing"})
                continue
            frame = pd.read_csv(path, low_memory=False)
            event_col = EVENT_COLS[endpoint]
            duration_col = DURATION_COLS[endpoint]
            if event_col not in frame.columns:
                cohort_rows.append({**base, "status": f"no {event_col}"})
                continue
            events = pd.to_numeric(frame[event_col], errors="coerce").fillna(0)
            durations = pd.to_numeric(frame.get(duration_col), errors="coerce")
            cohort_ids[(stratum, endpoint, landmark)] = set(
                pd.to_numeric(frame[ID_COL], errors="coerce").dropna().astype(int)
            )
            cohort_rows.append({
                **base,
                "n_patients": len(frame),
                "n_events": int(events.eq(1).sum()),
                "event_rate_pct": 100 * events.eq(1).mean(),
                "median_event_days": durations.loc[events.eq(1)].median(),
                "median_followup_days": durations.median(),
                "status": "ok",
            })

cohort_table = pd.DataFrame(cohort_rows)
display(cohort_table.round(1))

for _, row in cohort_table.loc[cohort_table["status"].eq("ok")].iterrows():
    if row["n_events"] < 50:
        print(
            f"!! {row['stratum']} / {row['endpoint']} @ {row['landmark_days']}d: "
            f"only {int(row['n_events'])} events; treat model estimates as underpowered."
        )

In [ ]:
# The two intent strata should be disjoint at every endpoint/landmark.
overlap_rows = []
for endpoint in ENDPOINTS:
    for landmark in LANDMARKS:
        loc = cohort_ids.get(("localized", endpoint, landmark))
        met = cohort_ids.get(("metastatic", endpoint, landmark))
        if loc is None or met is None:
            continue
        overlap_rows.append({
            "endpoint": endpoint,
            "landmark_days": landmark,
            "n_localized": len(loc),
            "n_metastatic": len(met),
            "n_overlap": len(loc & met),
        })
overlap_table = pd.DataFrame(overlap_rows)
if not overlap_table.empty:
    display(overlap_table)
    assert overlap_table["n_overlap"].eq(0).all(), "Intent-stratified cohorts overlap"

## 3. Univariate association comparison

Effects are joined on endpoint, landmark, and feature. `hr_ratio_met_vs_loc` is the metastatic hazard ratio divided by the localized hazard ratio. `p_heterogeneity` tests equality of the two log-HRs using standard errors reconstructed from their 95% confidence intervals; `q_heterogeneity` is BH-adjusted within each endpoint/landmark. This is an approximate between-stratum Wald comparison, not a replacement for a pooled Cox model with an explicit feature × intent interaction.

In [ ]:
def bh_adjust(values):
    p = pd.to_numeric(values, errors="coerce")
    out = pd.Series(np.nan, index=p.index, dtype=float)
    valid = p.dropna().sort_values()
    if valid.empty:
        return out
    adjusted = valid * len(valid) / np.arange(1, len(valid) + 1)
    adjusted = np.minimum.accumulate(adjusted.iloc[::-1])[::-1].clip(upper=1.0)
    out.loc[valid.index] = adjusted
    return out

univariate_frames = []
for endpoint in ENDPOINTS:
    for stratum in STRATA:
        for landmark in LANDMARKS:
            path = (tree(stratum, endpoint, "results") / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_univariate_nobs_adjusted.csv")
            if not require(path, f"{stratum}/{endpoint} univariate landmark {landmark}"):
                continue
            frame = pd.read_csv(path, low_memory=False)
            frame = frame.loc[frame["endpoint"].astype(str).str.lower().eq(endpoint)].copy()
            frame["endpoint"] = endpoint
            frame["stratum"] = stratum
            frame["landmark_days"] = landmark
            univariate_frames.append(frame)

if univariate_frames:
    univariate = pd.concat(univariate_frames, ignore_index=True)
else:
    univariate = pd.DataFrame()
    print("No univariate result files found.")

In [ ]:
association_comparison = pd.DataFrame()
if not univariate.empty:
    keys = ["endpoint", "landmark_days", "feature"]
    value_cols = [
        "lab_name", "feature_stat", "n_patients_used", "n_events_used",
        "coef_feature", "hazard_ratio_per_sd", "ci_lower", "ci_upper",
        "p_value", "q_value", "note",
    ]
    sides = {}
    for stratum in STRATA:
        side = univariate.loc[univariate["stratum"].eq(stratum)].copy()
        side = side[keys + [c for c in value_cols if c in side.columns]]
        side = side.drop_duplicates(keys)
        sides[stratum] = side
    association_comparison = sides["localized"].merge(
        sides["metastatic"], on=keys, how="outer",
        suffixes=("_localized", "_metastatic"), indicator=True,
    )
    for col in ("coef_feature", "hazard_ratio_per_sd", "ci_lower", "ci_upper", "p_value", "q_value"):
        for stratum in STRATA:
            name = f"{col}_{stratum}"
            if name in association_comparison:
                association_comparison[name] = pd.to_numeric(association_comparison[name], errors="coerce")

    association_comparison["delta_log_hr_met_minus_loc"] = (
        association_comparison["coef_feature_metastatic"] - association_comparison["coef_feature_localized"]
    )
    association_comparison["hr_ratio_met_vs_loc"] = np.exp(association_comparison["delta_log_hr_met_minus_loc"])
    association_comparison["same_direction"] = (
        np.sign(association_comparison["coef_feature_metastatic"])
        == np.sign(association_comparison["coef_feature_localized"])
    )
    for stratum in STRATA:
        lo = association_comparison[f"ci_lower_{stratum}"]
        hi = association_comparison[f"ci_upper_{stratum}"]
        association_comparison[f"se_log_hr_{stratum}"] = (np.log(hi) - np.log(lo)) / (2 * 1.96)
    se_delta = np.sqrt(
        association_comparison["se_log_hr_localized"] ** 2
        + association_comparison["se_log_hr_metastatic"] ** 2
    )
    association_comparison["z_heterogeneity"] = association_comparison["delta_log_hr_met_minus_loc"] / se_delta
    association_comparison["p_heterogeneity"] = association_comparison["z_heterogeneity"].abs().map(
        lambda z: math.erfc(z / math.sqrt(2)) if pd.notna(z) else np.nan
    )
    association_comparison["q_heterogeneity"] = association_comparison.groupby(
        ["endpoint", "landmark_days"], group_keys=False
    )["p_heterogeneity"].transform(bh_adjust)
    association_comparison["fdr_localized"] = association_comparison["q_value_localized"].lt(0.05)
    association_comparison["fdr_metastatic"] = association_comparison["q_value_metastatic"].lt(0.05)

    display(
        association_comparison.sort_values("p_heterogeneity")[
            keys + [
                "hazard_ratio_per_sd_localized", "hazard_ratio_per_sd_metastatic",
                "hr_ratio_met_vs_loc", "p_heterogeneity", "q_heterogeneity",
                "q_value_localized", "q_value_metastatic", "same_direction", "_merge",
            ]
        ].head(50).round(4)
    )

In [ ]:
if not association_comparison.empty:
    summary_rows = []
    for (endpoint, landmark), group in association_comparison.groupby(["endpoint", "landmark_days"]):
        both = group.loc[group["_merge"].eq("both")]
        summary_rows.append({
            "endpoint": endpoint,
            "landmark_days": landmark,
            "n_features_both": len(both),
            "n_same_direction": int(both["same_direction"].fillna(False).sum()),
            "spearman_log_hr": both[["coef_feature_localized", "coef_feature_metastatic"]].corr(method="spearman").iloc[0, 1],
            "n_fdr_localized": int(group["fdr_localized"].sum()),
            "n_fdr_metastatic": int(group["fdr_metastatic"].sum()),
            "n_heterogeneity_fdr": int(group["q_heterogeneity"].lt(0.05).sum()),
        })
    association_summary = pd.DataFrame(summary_rows)
    display(association_summary.round(3))

In [ ]:
# Log-HR concordance. Filled points have FDR q<0.05 in either stratum.
if not association_comparison.empty:
    fig, axes = plt.subplots(len(ENDPOINTS), len(LANDMARKS), figsize=(13, 11), squeeze=False)
    for i, endpoint in enumerate(ENDPOINTS):
        for j, landmark in enumerate(LANDMARKS):
            ax = axes[i, j]
            d = association_comparison.loc[
                association_comparison["endpoint"].eq(endpoint)
                & association_comparison["landmark_days"].eq(landmark)
            ].dropna(subset=["coef_feature_localized", "coef_feature_metastatic"])
            sig = d["fdr_localized"] | d["fdr_metastatic"]
            ax.scatter(d.loc[~sig, "coef_feature_localized"], d.loc[~sig, "coef_feature_metastatic"], s=12, alpha=0.35, color="#777777")
            ax.scatter(d.loc[sig, "coef_feature_localized"], d.loc[sig, "coef_feature_metastatic"], s=24, alpha=0.8, color="#6a3d9a")
            finite = d[["coef_feature_localized", "coef_feature_metastatic"]].to_numpy().ravel()
            finite = finite[np.isfinite(finite)]
            if finite.size:
                limit = max(abs(finite).max() * 1.08, 0.1)
                ax.plot([-limit, limit], [-limit, limit], ls="--", lw=1, color="black", alpha=0.5)
                ax.axhline(0, lw=0.5, color="0.7")
                ax.axvline(0, lw=0.5, color="0.7")
                ax.set_xlim(-limit, limit)
                ax.set_ylim(-limit, limit)
            ax.set_title(f"{endpoint} / +{landmark}d (n={len(d)})")
            if i == len(ENDPOINTS) - 1:
                ax.set_xlabel("Localized log HR / SD")
            if j == 0:
                ax.set_ylabel("Metastatic log HR / SD")
    fig.suptitle("Univariate effect concordance across ADT-intent strata", y=1.01)
    plt.tight_layout()
    plt.show()

## 4. Held-out multivariate performance

Metrics come only from each run’s held-out test set. Deltas are metastatic minus localized. Positive deltas favor metastatic for C-index/AUC; negative deltas favor metastatic for integrated Brier score. Because test cohorts differ, these are descriptive contrasts rather than paired tests.

In [ ]:
performance_frames = []
for endpoint in ENDPOINTS:
    for stratum, spec in STRATA.items():
        results_dir = tree(stratum, endpoint, "results")
        if not require(results_dir, f"{stratum}/{endpoint} results tree"):
            continue
        run = {
            "label": spec["label"],
            "output_dir": results_dir,
            "landmarks": list(LANDMARKS),
            "endpoint": endpoint,
        }
        frame = cp.summarize_outputs(run)
        frame["stratum"] = stratum
        performance_frames.append(frame)

performance = pd.concat(performance_frames, ignore_index=True) if performance_frames else pd.DataFrame()
if performance.empty:
    print("No multivariate result trees found.")
else:
    display(performance.round(4))

In [ ]:
performance_comparison = pd.DataFrame()
if not performance.empty:
    ok = performance.loc[performance["status"].eq("ok")].copy()
    index = ["endpoint", "landmark", "model", "config"]
    metrics = ["n_test", "n_test_events", "c_index", "mean_auc_t", "integrated_brier"]
    performance_comparison = ok.pivot_table(index=index, columns="stratum", values=metrics, aggfunc="first")
    performance_comparison.columns = [f"{metric}_{stratum}" for metric, stratum in performance_comparison.columns]
    performance_comparison = performance_comparison.reset_index()
    for metric in ("c_index", "mean_auc_t", "integrated_brier"):
        loc, met = f"{metric}_localized", f"{metric}_metastatic"
        if loc in performance_comparison and met in performance_comparison:
            performance_comparison[f"delta_{metric}_met_minus_loc"] = performance_comparison[met] - performance_comparison[loc]
    display(performance_comparison.round(4))

## 5. Missing artifacts

In [ ]:
if MISSING:
    unique_missing = list(dict.fromkeys(MISSING))
    print(f"{len(unique_missing)} artifact(s) were not found:")
    for item in unique_missing:
        print(f"  {item}")
    print("\nComplete the corresponding cells in 09_adt_intent_survival.ipynb, then rerun this notebook.")
else:
    print("All expected localized and metastatic artifacts were found.")